# SMRD - the full pipeline: lexical + RoBERTa + BiGCN, combined (Colab, T4 GPU)

This notebook needs nothing from earlier runs. On a Drive that already holds some of these folds (for example the
Twitter16 lexical and RoBERTa folds from `colab_text_long.ipynb`), those steps are skipped in a few seconds.

Before running: **Runtime > Change runtime type > T4 GPU > Save**.

The pipeline has three branches, trained on the same 5 folds, and one combination step:

* **lexical**: word counts + logistic regression (CPU).
* **text_long**: RoBERTa on the source tweet, up to 30 epochs.
* **graph**: BiGCN on the propagation tree.
* **stack_full**: weighted average of the three branches' class probabilities.

The validation fold is kept exactly as before: never trained on. It alone picks each branch's best model and the
combination weights. Each test fold is predicted once.
**Run the last cell before closing the tab or switching account.**

Results so far for reference (pooled accuracy):

| | Twitter16 | Twitter15 | PHEME |
|---|---|---|---|
| lexical | 0.848 | 0.840 | 0.871 |
| RoBERTa, up to 30 epochs | 0.784 | - | - |
| BiGCN | 0.804 | 0.797 | 0.858 |
| lexical + RoBERTa | 0.856 | - | - |

In [ ]:
#@title 1. Dependencies and GPU check
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NO GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the story-eval branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'story-eval'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (~3 min) - stop here if anything fails
!python -m pytest tests/test_cv.py tests/test_lexical_stack.py tests/test_encoder_pipeline.py tests/test_propagation.py -q

## twitter16

Run this one first and paste the output of 4d. On the Drive that already has the Twitter16 lexical and RoBERTa folds, only step c (BiGCN) does real work.

In [ ]:
#@title 4a. twitter16: lexical branch (CPU)
!python -m src.run_cv train --dataset twitter16 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: RoBERTa branch, up to 30 epochs (GPU)
!python -m src.run_cv train --dataset twitter16 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4c. twitter16: BiGCN branch (GPU)
!python -m src.run_cv train --dataset twitter16 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4d. twitter16: combine the branches, then compare
!python -m src.run_cv stack --dataset twitter16 --members lexical graph --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset twitter16 --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset twitter16 --members lexical text_long graph --tag full --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

Step b (RoBERTa) takes roughly 25 minutes on a T4.

In [ ]:
#@title 5a. twitter15: lexical branch (CPU)
!python -m src.run_cv train --dataset twitter15 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: RoBERTa branch, up to 30 epochs (GPU)
!python -m src.run_cv train --dataset twitter15 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5c. twitter15: BiGCN branch (GPU)
!python -m src.run_cv train --dataset twitter15 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5d. twitter15: combine the branches, then compare
!python -m src.run_cv stack --dataset twitter15 --members lexical graph --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset twitter15 --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset twitter15 --members lexical text_long graph --tag full --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme

The most expensive: step b can take an hour or more of GPU. Run it only if I ask for it.

In [ ]:
#@title 6a. pheme: lexical branch (CPU)
!python -m src.run_cv train --dataset pheme --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: RoBERTa branch, up to 30 epochs (GPU)
!python -m src.run_cv train --dataset pheme --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6c. pheme: BiGCN branch (GPU)
!python -m src.run_cv train --dataset pheme --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6d. pheme: combine the branches, then compare
!python -m src.run_cv stack --dataset pheme --members lexical graph --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset pheme --members lexical text_long --tag text --ckpt-root {CKPT_ROOT}
!python -m src.run_cv stack --dataset pheme --members lexical text_long graph --tag full --ckpt-root {CKPT_ROOT}
!python -m src.run_cv compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv compare --all-datasets --ckpt-root {CKPT_ROOT}
for p in glob.glob('results/cv/*_comparison.json'):
    shutil.copy(p, CKPT_ROOT)

In [ ]:
#@title 8. Flush Drive (run before closing the tab or switching account)
drive.flush_and_unmount()
print('Drive flushed and unmounted')